In [6]:
import mlflow 
import mlflow.sklearn   
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
import numpy as np
import os 
import dagshub
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [7]:
dagshub.init(repo_owner='shivamtiwari83032-collab', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow')


Initialized MLflow to track repo "shivamtiwari83032-collab/mlops-mini-project"

Repository shivamtiwari83032-collab/mlops-mini-project initialized!

In [8]:
df = pd.read_csv('https://raw.githubusercontent.com/shazizisazi/Datasets/a954a8e0efef2124456be0c386f4cb7979b928ad/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [9]:
import logging

logging.basicConfig(level=logging.INFO)

def lemmatization(text):
    try:
        lemmatizer = WordNetLemmatizer()
        return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])
    except Exception as e:
        logging.warning(f"Lemmatization failed: {e}")
        return text

def remove_stopwords(text):
    try:
        stop_words = set(stopwords.words('english'))
        return ' '.join([word for word in text.split() if word not in stop_words])
    except Exception as e:
        logging.warning(f"Stopword removal failed: {e}")
        return text

def remove_numbers(text):
    try:
        return ''.join([i for i in text if not i.isdigit()])
    except Exception as e:
        logging.warning(f"Number removal failed: {e}")
        return text

def lower_case(text):
    try:
        return ' '.join([word.lower() for word in text.split()])
    except Exception as e:
        logging.warning(f"Lowercase conversion failed: {e}")
        return text

def remove_punctuation(text):
    try:
        return text.translate(str.maketrans('', '', string.punctuation))
    except Exception as e:
        logging.warning(f"Punctuation removal failed: {e}")
        return text

def remove_urls(text):
    try:
        return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    except Exception as e:
        logging.warning(f"URL removal failed: {e}")
        return text

def remove_small_sentences(text):
    try:
        return ' '.join([word for word in text.split() if len(word) > 2])
    except Exception as e:
        logging.warning(f"Small word removal failed: {e}")
        return text

In [10]:
def normalize_text(df):
    try:
        if 'content' not in df.columns:
            raise ValueError("Column 'content' not found in dataframe")

        df['content'] = df['content'].astype(str)

        df['content'] = df['content'].apply(lemmatization)
        df['content'] = df['content'].apply(remove_stopwords)
        df['content'] = df['content'].apply(remove_numbers)
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_punctuation)
        df['content'] = df['content'].apply(remove_urls)
        df['content'] = df['content'].apply(remove_small_sentences)

        logging.info("Text normalization completed successfully")
        return df
    
    except Exception as e:
        logging.error(f"Error in text normalization: {e}")
        raise

In [11]:
normalize_text(df)
df.head()

INFO:root:Text normalization completed successfully


,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [13]:
x=df['sentiment'].isin(['happiness','sadness'])
df=df[x]

In [14]:
df['sentiment']=df['sentiment'].replace({'sadness':0,'happiness':1})
df.sample(5)

C:\Users\pc\AppData\Local\Temp\ipykernel_52028\2069291767.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment']=df['sentiment'].replace({'sadness':0,'happiness':1})


,sentiment,content
28851,1,had nice pre mothers day dinner out now cockta...
19508,1,passed driver test driveoh wait turn till october
2921,0,going get full night sleep tonight arm get bet...
9289,0,wish sun would shine
23714,1,elephatt lol thanks you nice meeting aswell lo...


In [15]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']

xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)   

In [16]:
mlflow.set_experiment('LOR hyperparameter tunning')

2026/04/16 20:46:21 INFO mlflow.tracking.fluent: Experiment with name 'LOR hyperparameter tunning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ff08e2f6579c4e8b85b664f24ce7107e', creation_time=1776352581477, experiment_id='2', last_update_time=1776352581477, lifecycle_stage='active', name='LOR hyperparameter tunning', tags={}, trace_location=None, workspace='default'>

In [17]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}


In [18]:
from sklearn.model_selection import GridSearchCV

In [19]:
with mlflow.start_run(run_name='LOR_BOW_HP'):
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, n_jobs=-1)
    grid_search.fit(xtrain, ytrain)

    for params,mean_score,std_score in zip(grid_search.cv_results_['params'], grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['std_test_score']):
        with mlflow.start_run(run_name='lr with params ', nested=True):
            model=LogisticRegression(**params)
            model.fit(xtrain, ytrain)

            ypred=model.predict(xtest)
            acc=accuracy_score(ytest,ypred) 
            prec=precision_score(ytest,ypred)
            rec=recall_score(ytest,ypred)
            f1=f1_score(ytest,ypred)
            mlflow.log_params(params)
            mlflow.log_metric('accuracy', acc)
            mlflow.log_metric('precision', prec)
            mlflow.log_metric('recall', rec)
            mlflow.log_metric('f1_score', f1)
            mlflow.sklearn.log_model(model, 'model')
            mlflow.log_metric('mean_test_score', mean_score )
            mlflow.log_metric('std_test_score', std_score )


            print(f"Params: {params}, Accuracy: {acc}, Precision: {prec}, Recall: {rec}, F1 Score: {f1}, Mean Test Score: {mean_score}, Std Test Score: {std_score}")



    best_parms=grid_search.best_params_
    best_score=grid_search.best_score_
    mlflow.log_metric('best_score', best_score)
    mlflow.log_params(best_parms)

    print(f"Best Parameters: {best_parms}, Best Score: {best_score}")

2026/04/16 21:03:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 21:03:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Params: {'C': 0.01, 'penalty': 'l1', 'solver': 'liblinear'}, Accuracy: 0.6, Precision: 0.8291814946619217, Recall: 0.22955665024630542, F1 Score: 0.3595679012345679, Mean Test Score: 0.5598265757423909, Std Test Score: 0.006465448689253014
🏃 View run lr with params  at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2/runs/f815c17009a84d42823b17e06553b8de
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2


2026/04/16 21:03:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 21:03:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Params: {'C': 0.01, 'penalty': 'l2', 'solver': 'liblinear'}, Accuracy: 0.7556626506024097, Precision: 0.7597137014314929, Recall: 0.7320197044334975, F1 Score: 0.745609633718013, Mean Test Score: 0.7535854085419434, Std Test Score: 0.009614829774509668
🏃 View run lr with params  at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2/runs/4ff174c23497467a89f926bbcf39367f
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2


2026/04/16 21:03:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 21:04:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

Params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}, Accuracy: 0.7865060240963856, Precision: 0.7734225621414914, Recall: 0.7970443349753694, F1 Score: 0.7850557981562348, Mean Test Score: 0.770454694002048, Std Test Score: 0.005329391098259723
🏃 View run lr with params  at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2/runs/a25d3dbbb99a46ea87816f71fa5b9492
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2


2026/04/16 21:06:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 21:06:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Params: {'C': 100, 'penalty': 'l1', 'solver': 'liblinear'}, Accuracy: 0.7580722891566265, Precision: 0.7588294651866802, Recall: 0.7408866995073892, F1 Score: 0.7497507477567298, Mean Test Score: 0.7503325417401977, Std Test Score: 0.007453232550252428
🏃 View run lr with params  at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2/runs/d4342ccdbc2346e2bc630e3ae0380210
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2


2026/04/16 21:06:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 21:07:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Params: {'C': 100, 'penalty': 'l2', 'solver': 'liblinear'}, Accuracy: 0.775421686746988, Precision: 0.7626794258373206, Recall: 0.7852216748768472, F1 Score: 0.7737864077669903, Mean Test Score: 0.7602123503053806, Std Test Score: 0.007299580232067395
🏃 View run lr with params  at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2/runs/0f0a5b6864c84f4087e4cc99fbdc2ab3
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2
Best Parameters: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}, Best Score: 0.7817817381642301
🏃 View run LOR_BOW_HP at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2/runs/0a7315a079834532a63b6d42d737105f
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/2
